In [ ]:
!pip install -q "transformers[torch]>=4.46,<5" datasets jiwer

In [2]:
!pip install gdown

In [3]:
import os
lst_file_names = []
for file_name in os.listdir('/kaggle/input/ocrtest/new_public_test'):
    lst_file_names.append(file_name)
with open('test_gt.txt', 'w') as f:
    for f_name in lst_file_names:
        f.write(f_name+'\n')

In [4]:
import pandas as pd

df = pd.read_fwf('/kaggle/working/test_gt.txt', header=None)
df.rename(columns={0: "file_name"}, inplace=True)
df

,file_name
0,public_test_img_23072.png
1,public_test_img_6326.jpg
2,public_test_img_774.jpg
3,public_test_img_28214.png
4,public_test_img_13609.jpg
...,...
32995,public_test_img_23743.png
32996,public_test_img_26551.png
32997,public_test_img_10242.jpg
32998,public_test_img_17813.png


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text 
        file_name = self.df['file_name'][idx]
        #text = self.df['text'][idx]
        # some file names end with jp instead of jpg, the two lines below fix this
        if file_name.endswith('jp'):
          file_name = file_name + 'g'
        # prepare image (i.e. resize + normalize)
        image = Image.open(self.root_dir + file_name).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        # important: make sure that PAD tokens are ignored by the loss function

        encoding = {"file_name":file_name,"pixel_values": pixel_values.squeeze()}
        return encoding

In [ ]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
test_dataset = IAMDataset(root_dir='/kaggle/input/ocrtest/new_public_test/',
                           df=df,
                           processor=processor)

In [7]:
from transformers import VisionEncoderDecoderModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionEncoderDecoderModel.from_pretrained("/kaggle/input/ckpt4000/ckpt-4000")
model.to(device)

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_fea

In [ ]:
from tqdm.notebook import tqdm

def get_submission(df, root_dir):
    submission = pd.DataFrame()
    f_name = []
    text = []
    out_lst = []
    for file_name in tqdm(df['file_name']):
        if file_name.endswith('jp'):
          file_name = file_name + 'g'
        # prepare image (i.e. resize + normalize)
        image = Image.open(os.path.join(root_dir,file_name)).convert("RGB")
        pixel_values = processor(image, return_tensors="pt").pixel_values
        # important: make sure that PAD tokens are ignored by the loss function
        pixel_values = pixel_values.to(device)
        outputs = model.generate(pixel_values)
        pred_str = processor.batch_decode(outputs, skip_special_tokens=True)
        out = {'file_name': file_name, 'text': pred_str[0]}
        out_lst.append(out)
    return out_lst

In [9]:
lst_out = get_submission(df= df, root_dir= '/kaggle/input/ocrtest/new_public_test')

out_df = pd.DataFrame(lst_out)
out_df

  0%|          | 0/33000 [00:00<?, ?it/s]

,file_name,text
0,public_test_img_23072.png,[nhơ]
1,public_test_img_6326.jpg,[thuê]
2,public_test_img_774.jpg,[trát]
3,public_test_img_28214.png,[chính]
4,public_test_img_13609.jpg,[SÁNH]
...,...,...
32995,public_test_img_23743.png,[bản]
32996,public_test_img_26551.png,[manh]
32997,public_test_img_10242.jpg,[côngectô]
32998,public_test_img_17813.png,[bán]


In [10]:
out_df.to_csv('/kaggle/working/out.csv', index=False)